# Stage 3 — Feature engineering

Calendar, lag, rolling-window, price, and categorical-encoding features — built one group at a time, with leakage risks called out explicitly.

## Step 1: Calendar features

Deriving is_weekend, snap_active (vectorized), and has_event on top of
the raw calendar columns from Stage 0. Zero leakage risk here — every
one of these is fully knowable in advance for any future date, since
none of them touch the sales/target column.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/Users/asadkhan.pathan/Documents/New project/m5 forecasting ")
df = pd.read_parquet(BASE_DIR / "data" / "processed" / "sales_long.parquet")
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# wday: 1=Saturday, 2=Sunday, ..., 7=Friday (per M5's calendar convention,
# confirmed in Stage 1). So weekend = wday 1 or 2.
df["is_weekend"] = df["wday"].isin([1, 2]).astype("int8")

# Vectorized version of the state-specific SNAP lookup we did slowly with
# .apply() in Stage 1 — np.select checks each state condition in order and
# picks the matching snap column's value, all in one array operation.
df["snap_active"] = np.select(
    [df["state_id"] == "CA", df["state_id"] == "TX", df["state_id"] == "WI"],
    [df["snap_CA"], df["snap_TX"], df["snap_WI"]],
).astype("int8")

df["has_event"] = (df["event_name_1"].notna() | df["event_name_2"].notna()).astype("int8")

print(df[["date", "wday", "is_weekend", "state_id", "snap_active", "event_name_1", "has_event"]].sample(10, random_state=1))
print(df[["is_weekend", "snap_active", "has_event"]].mean())


               date  wday  is_weekend state_id  snap_active   event_name_1  \
49519895 2014-02-05     5           0       WI            1            NaN   
16763330 2013-06-01     1           1       CA            1            NaN   
45862025 2011-04-16     1           1       WI            0            NaN   
9106686  2015-01-23     7           0       CA            0            NaN   
32293748 2014-09-21     2           1       TX            0            NaN   
44826815 2014-10-21     4           0       WI            0            NaN   
38406572 2011-02-03     6           0       TX            1            NaN   
6550083  2014-03-23     2           1       CA            0            NaN   
12878784 2011-10-05     5           0       CA            1            NaN   
58901973 2012-02-20     3           0       WI            0  PresidentsDay   

          has_event  
49519895          0  
16763330          0  
45862025          0  
9106686           0  
32293748          0  
44826815 

In [2]:
df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,is_weekend,snap_active,has_event
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,NaN,1,0,0
1,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0,2011-01-30,11101,...,NaN,NaN,NaN,0,0,0,NaN,1,0,0
2,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0,2011-01-31,11101,...,NaN,NaN,NaN,0,0,0,NaN,0,0,0
3,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_4,0,2011-02-01,11101,...,NaN,NaN,NaN,1,1,0,NaN,0,1,0
4,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_5,0,2011-02-02,11101,...,NaN,NaN,NaN,1,0,1,NaN,0,1,0


## Step 2: Lag features (28+ days back, not lag-1)

M5 needs one model to forecast all 28 horizon days at once, so every
lag feature must be usable for the FARTHEST day of the horizon, not
just the nearest. That means lags of 28+ days, not lag-1/lag-7 — those
would leak, since they aren't actually available by day 28 of a real
forecast.


In [3]:
LAG_DAYS = [28, 35, 42, 49, 56]

for lag in LAG_DAYS:
    df[f"lag_{lag}"] = df.groupby("id", observed=True)["sales"].shift(lag)

print(df[["id", "date", "sales"] + [f"lag_{lag}" for lag in LAG_DAYS]].iloc[25:35])
print(df[[f"lag_{lag}" for lag in LAG_DAYS]].isna().sum())


                               id       date  sales  lag_28  lag_35  lag_42  \
25  HOBBIES_1_001_CA_1_evaluation 2011-02-23      0     NaN     NaN     NaN   
26  HOBBIES_1_001_CA_1_evaluation 2011-02-24      0     NaN     NaN     NaN   
27  HOBBIES_1_001_CA_1_evaluation 2011-02-25      0     NaN     NaN     NaN   
28  HOBBIES_1_001_CA_1_evaluation 2011-02-26      0     0.0     NaN     NaN   
29  HOBBIES_1_001_CA_1_evaluation 2011-02-27      0     0.0     NaN     NaN   
30  HOBBIES_1_001_CA_1_evaluation 2011-02-28      0     0.0     NaN     NaN   
31  HOBBIES_1_001_CA_1_evaluation 2011-03-01      0     0.0     NaN     NaN   
32  HOBBIES_1_001_CA_1_evaluation 2011-03-02      0     0.0     NaN     NaN   
33  HOBBIES_1_001_CA_1_evaluation 2011-03-03      0     0.0     NaN     NaN   
34  HOBBIES_1_001_CA_1_evaluation 2011-03-04      0     0.0     NaN     NaN   

    lag_49  lag_56  
25     NaN     NaN  
26     NaN     NaN  
27     NaN     NaN  
28     NaN     NaN  
29     NaN     NaN  
30  

## Step 3: Rolling window features (built on lag_28, not raw sales)

Rolling mean of the LAGGED series, not raw sales — this is the leakage
trap the plan warned about. Rolling raw sales would use the most recent
actual days, which isn't valid for the far end of a 28-day horizon.
Rolling the already-shifted lag_28 series keeps every window safely in
the past relative to the whole forecast horizon.
"

In [4]:
# Reuse the lag_28 column as the base for rolling stats, so every
# rolling window is automatically at least 28 days old.
df["sales_lag28"] = df.groupby("id", observed=True)["sales"].shift(28)

for window in [7, 28]:
    df[f"rolling_mean_{window}_lag28"] = (
        df.groupby("id", observed=True)["sales_lag28"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

cols = ["id", "date", "sales", "sales_lag28", "rolling_mean_7_lag28", "rolling_mean_28_lag28"]
print(df[cols].iloc[25:35])
print(df[["rolling_mean_7_lag28", "rolling_mean_28_lag28"]].isna().sum())


                               id       date  sales  sales_lag28  \
25  HOBBIES_1_001_CA_1_evaluation 2011-02-23      0          NaN   
26  HOBBIES_1_001_CA_1_evaluation 2011-02-24      0          NaN   
27  HOBBIES_1_001_CA_1_evaluation 2011-02-25      0          NaN   
28  HOBBIES_1_001_CA_1_evaluation 2011-02-26      0          0.0   
29  HOBBIES_1_001_CA_1_evaluation 2011-02-27      0          0.0   
30  HOBBIES_1_001_CA_1_evaluation 2011-02-28      0          0.0   
31  HOBBIES_1_001_CA_1_evaluation 2011-03-01      0          0.0   
32  HOBBIES_1_001_CA_1_evaluation 2011-03-02      0          0.0   
33  HOBBIES_1_001_CA_1_evaluation 2011-03-03      0          0.0   
34  HOBBIES_1_001_CA_1_evaluation 2011-03-04      0          0.0   

    rolling_mean_7_lag28  rolling_mean_28_lag28  
25                   NaN                    NaN  
26                   NaN                    NaN  
27                   NaN                    NaN  
28                   NaN                    NaN  
2

## Step 4: Price features

Unlike sales, price is a known INPUT (retailers set prices in advance),
so price features don't need the lag-28 treatment sales features
needed. The real risk here is different: aggregating price over a
series' ENTIRE history (including future/test rows) would leak future
price trends. Fix: use an expanding, backward-only average per row.


In [6]:
df["price_change_vs_last_week"] = df["sell_price"] - df.groupby("id", observed=True)["sell_price"].shift(7)

# Expanding mean = "average of everything up to and including yesterday"
# for this row's own date — shifted by 1 so a row never includes its own
# price in its own historical average.
expanding_avg_price = (
    df.groupby("id", observed=True)["sell_price"]
    .apply(lambda s: s.shift(1).expanding().mean())
    .reset_index(level=0, drop=True)
)
df["price_rel_to_own_avg"] = df["sell_price"] / expanding_avg_price

cols = ["id", "date", "sell_price", "price_change_vs_last_week", "price_rel_to_own_avg"]
print(df[cols].dropna().sample(16, random_state=1))
print(df[["price_change_vs_last_week", "price_rel_to_own_avg"]].isna().sum())






                                       id       date  sell_price  \
3305067       FOODS_1_170_CA_1_evaluation 2015-02-22        5.48   
45403572      FOODS_2_222_WI_1_evaluation 2015-07-28        2.57   
8238790   HOUSEHOLD_2_099_CA_2_evaluation 2014-04-29        5.94   
55351140  HOUSEHOLD_1_520_WI_3_evaluation 2015-06-01        5.48   
27052503      FOODS_1_132_TX_1_evaluation 2013-03-25        1.78   
13717943  HOUSEHOLD_1_414_CA_3_evaluation 2013-07-13        0.97   
36578343    HOBBIES_2_136_TX_3_evaluation 2011-08-15        2.58   
33748077      FOODS_2_315_TX_2_evaluation 2016-02-23        2.47   
15495660      FOODS_2_088_CA_3_evaluation 2012-11-16        6.97   
10046485      FOODS_2_300_CA_2_evaluation 2016-01-13        6.48   
19548266  HOUSEHOLD_1_367_CA_4_evaluation 2012-04-28        3.97   
25357817  HOUSEHOLD_1_310_TX_1_evaluation 2012-09-13        7.97   
36659497  HOUSEHOLD_1_028_TX_3_evaluation 2015-12-05        2.97   
37465048  HOUSEHOLD_1_452_TX_3_evaluation 2016-0

## Step 5: Categorical encoding

Converting category columns to integer codes for LightGBM's native
categorical split-finding — not one-hot, which would explode into
thousands of sparse columns for high-cardinality fields like item_id.
This is a pure identity mapping (no target involved), so unlike our
other features, it carries zero leakage risk regardless of train/test
split.


In [7]:
cat_features = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
]

for col in cat_features:
    df[f"{col}_enc"] = df[col].cat.codes

print(df[cat_features + [f"{c}_enc" for c in cat_features]].sample(5, random_state=1))

# Sanity check: number of distinct codes should match number of distinct
# category labels for each column.
for col in cat_features:
    print(col, "unique labels:", df[col].nunique(), " unique codes:", df[f"{col}_enc"].nunique())


                  item_id      dept_id     cat_id store_id state_id  \
49519895  HOUSEHOLD_2_024  HOUSEHOLD_2  HOUSEHOLD     WI_2       WI   
16763330      FOODS_3_314      FOODS_3      FOODS     CA_3       CA   
45862025      FOODS_3_061      FOODS_3      FOODS     WI_1       WI   
9106686       FOODS_1_085      FOODS_1      FOODS     CA_2       CA   
32293748  HOUSEHOLD_2_258  HOUSEHOLD_2  HOUSEHOLD     TX_2       TX   

         event_name_1 event_type_1 event_name_2 event_type_2  item_id_enc  \
49519895          NaN          NaN          NaN          NaN         1120   
16763330          NaN          NaN          NaN          NaN         2538   
45862025          NaN          NaN          NaN          NaN         2285   
9106686           NaN          NaN          NaN          NaN         1619   
32293748          NaN          NaN          NaN          NaN         1353   

          dept_id_enc  cat_id_enc  store_id_enc  state_id_enc  \
49519895            3           1            